In [ ]:
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch
import gc
import gradio as gr

In [ ]:
import os
hf_token = os.getenv('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [ ]:
LLAMA = "meta-llama/Llama-3.2-1B-Instruct"
GEMMA = "google/gemma-3-270m-it"
DEEPSEEK = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

In [ ]:
system_prompt="""
You are a database engineer who is experienced in ETL and skeptical about data sharing and privacy. You are an assistant
that when given details about the data to be shared, you will generate synthetic data that closely mimics the user request
with certainty that the given data are exactly what they wanted. The user will not share data. you must generate data. 

**Some points**

- Column names are explanatory based on the data they will contain
- Every data record should have a unique id
- The data should be presented in markdown table format
"""

In [ ]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

In [ ]:
# 4-bit quantization only when bitsandbytes is installed (e.g. on Linux/CUDA).
try:
    import bitsandbytes
    use_4bit = torch.cuda.is_available()
except Exception:
    use_4bit = False

if use_4bit:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float32,
        bnb_4bit_quant_type="nf4"
    )
    print("Using 4-bit quantization (bitsandbytes).")
else:
    quant_config = None
    print("Using full precision (no bitsandbytes / non-CUDA).")

In [ ]:
# Initialize tokenizer and model globally to avoid repeated loading
# These will be loaded once when the cell is executed
tokenizer = None
model_instance = None

def generate_synthetic_data(msg, model_name, stream_to_console=True):
  global tokenizer, model_instance

  if tokenizer is None or model_instance is None or model_instance.name_or_path != model_name:
    # Clear memory if a different model is being loaded or if it's the first load
    if model_instance is not None:
      del model_instance
    if tokenizer is not None:
      del tokenizer
    gc.collect()
    if device == "cuda":
      torch.cuda.empty_cache()

    print(f"Loading model {model_name}...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    load_kwargs = {"torch_dtype": torch.float32}
    if quant_config is not None:
        load_kwargs["quantization_config"] = quant_config
    model_instance = AutoModelForCausalLM.from_pretrained(model_name, **load_kwargs).to(device)
    print("Model loaded.")

  messages =[{
      "role": "system",
      "content": system_prompt},
    {
      "role": "user",
      "content": msg
  }]

  input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt").to(device)
  # Calculate the length of the prompt
  prompt_length = input_ids.shape[1]
  attention_mask = torch.ones_like(input_ids, dtype=torch.long, device=device)
  # Only stream to terminal when not called from Gradio
  streamer = TextStreamer(tokenizer, skip_special_tokens=True, skip_prompt=True) if stream_to_console else None

  outputs = model_instance.generate(
      input_ids=input_ids,
      attention_mask=attention_mask,
      max_new_tokens=1024*4,
      min_new_tokens=32,  # avoid empty reply (model sometimes outputs EOS immediately)
      pad_token_id=tokenizer.pad_token_id,
      eos_token_id=tokenizer.eos_token_id,
      do_sample=True,
      temperature=0.7,
      top_p=0.9,
      streamer=streamer,
  )

  # Decode only the newly generated tokens
  generated_tokens = outputs[0][prompt_length:]
  return tokenizer.decode(generated_tokens, skip_special_tokens=True)

## User Interface to interact with the models

In [ ]:
# Map dropdown labels to HuggingFace model IDs (must match LLAMA, GEMMA, DEEPSEEK from above)
MODEL_IDS = {
    "Llama-3.2-1B-Instruct": LLAMA,
    "Gemma-3-270m-it": GEMMA,
    "DeepSeek-R1-Distill-Qwen-1.5B": DEEPSEEK,
}


def create_interface(generate_fn):
    """Build the Gradio UI. Pass generate_synthetic_data so it's defined when this runs."""
    def on_generate(msg, model_label):
        model_id = MODEL_IDS.get(model_label, model_label)
        # Don't stream to terminal; output only in the Gradio textbox when user clicks Generate
        return generate_fn(msg, model_id, stream_to_console=False)

    with gr.Blocks(title="Synthetic Dataset Generator", theme=gr.themes.Monochrome()) as demo:
        gr.Markdown("Generate a dataset of synthetic data based on the user request.")

        with gr.Row():
            with gr.Column():
                user_message = gr.Textbox(
                    label="User request",
                    placeholder="e.g. generate sensors data from the thermometer, humidity sensor, and water levels.",
                    value="generate sensors data from the thermometer, humidity sensor, and water levels.",
                    info="User request",
                )
                model_name = gr.Dropdown(
                    choices=list(MODEL_IDS.keys()),
                    value="Llama-3.2-1B-Instruct",
                    label="Model",
                    info="Model to use",
                )

                generate_btn = gr.Button(
                    "Generate dataset", variant="primary", size="lg"
                )

            with gr.Column():
                output_text = gr.Textbox(
                    label="Generated dataset",
                    lines=20,
                    placeholder="Generated list will appear here...",
                    show_copy_button=True,
                )

        gr.Markdown(
            """
        ### About
        - **Model**: Loaded with 4-bit quantization when available (e.g. CUDA); otherwise full precision.
        - **Made with**: Gradio
        """
        )

        generate_btn.click(
            fn=on_generate,
            inputs=[user_message, model_name],
            outputs=output_text,
        )

        return demo

In [ ]:
# Run the cell that defines generate_synthetic_data before this cell.
demo = create_interface(generate_synthetic_data)
demo.launch()